# Анализ ДТП в Великобритании

STATS19 Collisions: одна запись — зарегистрированное ДТП с пострадавшими. Архитектура: `CSV → ETL → PostgreSQL → Jupyter → Plotly / Superset`. PostgreSQL хранит восемь используемых полей за все годы; годовая динамика использует всю таблицу, остальные расчёты — 2022–2024.

**Запуск:** откройте ноутбук в корне проекта рядом с `.env`, запустите Docker Desktop, выполните `docker compose up -d postgres` и `python etl.py`, дождитесь завершения ETL. Выберите Python-окружение, перезапустите ядро и выполните все ячейки. Первая ячейка устанавливает отсутствующие пакеты, поэтому может потребоваться интернет. Все результаты рассчитываются на вашей БД и сохраняются в `results/`.

[README](README.md) содержит подготовку Python, SQL-схемы и PostgreSQL 17, расположение исходного CSV, устройство ETL, Docker volume, настройку и экспорт Superset. Ноутбук проверяет ETL, но не запускает его повторно. При отсутствии отчёта полноту загрузки необходимо подтвердить отдельно.

| Участник | Ответственность |
|---|---|
| Иван Ф | PostgreSQL, Docker и окружение |
| Артём | ETL, очистка и контроль качества |
| Иван А | Статистические проверки, регрессия и классификация |
| Кира | Гипотезы, графики, Superset и выводы |

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn', 'scipy': 'scipy', 'sqlalchemy': 'sqlalchemy',
    'psycopg': 'psycopg[binary]', 'dotenv': 'python-dotenv',
    'plotly': 'plotly', 'nbformat': 'nbformat',
}

missing = []
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    print('Установка:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Python:', sys.executable)


In [ ]:
import json
import os
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
from scipy.stats import norm, ttest_1samp
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)

ROOT = None
for folder in [Path.cwd(), Path.cwd() / 'road-safety-big-data']:
    if (folder / '.env').is_file():
        ROOT = folder
        break
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук в папке проекта рядом с .env.')
load_dotenv(ROOT / '.env', override=True)
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
START_YEAR = 2022
END_YEAR = 2024
ALPHA = 0.05
RANDOM_STATE = 42
pio.renderers.default = 'notebook_connected'

versions = {}
for package in packages.values():
    distribution = package.split('[')[0]
    versions[distribution] = version(distribution)
(RESULTS / 'analysis_environment.json').write_text(
    json.dumps({'python': sys.version, 'packages': versions}, indent=2),
    encoding='utf-8',
)
print('Папка проекта:', ROOT)
print('Период исследования:', START_YEAR, '–', END_YEAR)

def finish_figure(fig, filename, show=False):
    fig.tight_layout()
    fig.savefig(RESULTS / filename, dpi=150)
    if show:
        plt.show()
    plt.close(fig)


## 1. Подключение и контроль загрузки

Пароль читается из `.env` и не выводится. Проверяются статус и баланс последнего ETL, количество строк в БД, размер доступного CSV и согласованность годовой статистики. Точную версию и дату скачивания CSV зафиксируйте в README; маленькая выборка не заменяет загрузку полного набора.

In [ ]:
required_env = ['POSTGRES_DB', 'POSTGRES_USER', 'POSTGRES_PASSWORD']
absent = [name for name in required_env if not os.getenv(name)]
if absent:
    raise ValueError('Заполните .env: ' + ', '.join(absent))

reports = sorted(RESULTS.glob('etl_report_*.json'))
etl_report = None
if reports:
    etl_report = json.loads(reports[-1].read_text(encoding='utf-8'))
    if etl_report.get('status') != 'completed':
        raise RuntimeError('Последний ETL не завершён успешно. Проверьте ' + reports[-1].name)
    display(pd.DataFrame({
        'Показатель': ['Прочитано', 'Добавлено', 'Повторов', 'Отклонено'],
        'Значение': [etl_report[k] for k in [
            'rows_read', 'rows_inserted', 'rows_duplicate', 'rows_rejected']],
    }))
    counted = sum(etl_report[k] for k in [
        'rows_inserted', 'rows_duplicate', 'rows_rejected'])
    if counted != etl_report['rows_read']:
        raise ValueError('Нарушен баланс строк в отчёте ETL.')
else:
    print('Отчёт ETL отсутствует. Полноту загрузки нужно подтвердить отдельно.')

url = URL.create(
    'postgresql+psycopg', username=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'], host='127.0.0.1',
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    database=os.environ['POSTGRES_DB'],
)
engine = create_engine(url, pool_pre_ping=True, connect_args={'connect_timeout': 10})
try:
    with engine.connect() as conn:
        total = conn.execute(text('SELECT COUNT(*) FROM public.collisions')).scalar_one()
except Exception:
    raise RuntimeError(
        'Не удалось прочитать public.collisions. Проверьте Docker Desktop, '
        'docker compose ps, .env и создание таблицы sql/schema.sql.'
    ) from None
if total == 0:
    raise ValueError('Таблица пуста. Выполните python etl.py и дождитесь завершения.')
if etl_report and total != etl_report.get('rows_in_database'):
    raise ValueError('Число строк в БД отличается от последнего отчёта ETL.')
print(f'Подключение работает. Записей в БД: {total:,}')


In [ ]:
SOURCE = ROOT / 'data/raw/dft-road-casualty-statistics-collision-1979-latest-published-year.csv'
if SOURCE.is_file():
    source_bytes = SOURCE.stat().st_size
    print(f'Исходный файл: {source_bytes / 10**9:.2f} ГБ ({source_bytes / 1024**3:.2f} ГиБ)')
    if source_bytes <= 500_000_000:
        print('Проверьте требование задания: размер набора должен быть больше 0,5 ГБ.')
    if etl_report and source_bytes != etl_report.get('source_bytes'):
        raise ValueError('Размер текущего CSV отличается от файла последнего ETL.')
else:
    print('CSV локально отсутствует. Анализ использует БД; размер источника проверьте по отчёту ETL.')

with engine.connect() as conn:
    annual = pd.read_sql_query(text("""
        SELECT collision_year AS year, COUNT(*) AS collision_count
        FROM public.collisions GROUP BY collision_year ORDER BY collision_year
    """), conn)
    df = pd.read_sql_query(text("""
        SELECT collision_index, collision_year, collision_severity,
               speed_limit, light_conditions, date, time, police_force
        FROM public.collisions
        WHERE collision_year BETWEEN :start_year AND :end_year
    """), conn, params={'start_year': START_YEAR, 'end_year': END_YEAR})
if df.empty:
    raise ValueError('Нет данных за выбранный период.')
if int(annual['collision_count'].sum()) != total:
    raise ValueError('Годовая статистика не согласуется с общим количеством строк.')
annual.to_csv(RESULTS / 'annual.csv', index=False)
print(f'Записей за выбранный период: {len(df):,}')
display(df.head())
display(annual.tail(10))


## 2. Признаки и качество

`severe`: 1 для тяжести 1–2, 0 для кода 3, пропуск для неизвестной тяжести. Лимит скорости — mph; освещение: 1 — день, 6 — темнота без освещения. Неизвестные значения исключаются только из расчётов, которым соответствующий признак необходим. Полицейское подразделение — категория. Контроль ниже относится к выбранному периоду; качество полной загрузки описывает отчёт ETL.

In [ ]:
if df['collision_index'].isna().any() or df['collision_index'].duplicated().any():
    raise ValueError('Нарушена уникальность или заполненность идентификаторов.')
for column in ['collision_year', 'collision_severity', 'speed_limit', 'light_conditions', 'police_force']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].where(df['date'].dt.year.eq(df['collision_year']))
time_text = df['time'].astype('string')

time_pattern = r'(?:[01]\d|2[0-3]):[0-5]\d(?::[0-5]\d(?:\.\d+)?)?'
valid_time = time_text.str.fullmatch(time_pattern).fillna(False)
df['hour'] = pd.to_numeric(time_text.where(valid_time).str[:2], errors='coerce')
df['weekday'] = df['date'].dt.dayofweek

df['severe'] = df['collision_severity'].map({1: 1.0, 2: 1.0, 3: 0.0})
quality = pd.DataFrame({
    'metric': ['rows_in_database', 'rows_in_period', 'unknown_severity',
               'unknown_or_other_speed', 'unknown_or_other_light', 'invalid_date_or_time'],
    'value': [total, len(df), int(df['severe'].isna().sum()),
              int((~df['speed_limit'].isin([20, 30, 40, 50, 60, 70])).sum()),
              int((~df['light_conditions'].isin([1, 4, 5, 6, 7])).sum()),
              int((df['date'].isna() | df['hour'].isna()).sum())],
})
quality.to_csv(RESULTS / 'quality.csv', index=False)
display(quality)


## 3. Три гипотезы

- **H1:** доля тяжёлых/смертельных ДТП выше при лимите 50–70 mph, чем 20–30 mph.
- **H2:** эта доля выше в темноте без освещения, чем днём.
- **H3:** ДТП в расчёте на час больше в 07:00–09:59 и 16:00–18:59, чем в остальные часы.

Для H1/H2 — односторонний Z-тест двух долей; для H3 — односторонний t-тест парных дневных разностей (6 часов пик и 18 остальных). Нулевая гипотеза допускает равенство или отсутствие увеличения; α = 0,05, поправка Бонферрони на три проверки. Пропущенные дни не считаются нулевыми. Оценивайте также размер различий: пространственная/временная зависимость влияет на значимость, причинность не устанавливается.

In [ ]:
def compare_shares(data, mask1, mask0, name, label1, label0):

    a = data.loc[mask1, 'severe'].dropna()
    b = data.loc[mask0, 'severe'].dropna()
    n1 = len(a)
    n0 = len(b)
    x1 = int(a.sum())
    x0 = int(b.sum())
    if min(x1, n1 - x1, x0, n0 - x0) < 5:
        raise ValueError(f'{name}: недостаточно наблюдений для Z-теста.')
    p1 = x1 / n1
    p0 = x0 / n0
    pooled = (x1 + x0) / (n1 + n0)
    standard_error = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n0))
    z = (p1 - p0) / standard_error
    table = pd.DataFrame({
        'group': [label1, label0], 'total': [n1, n0],
        'severe_count': [x1, x0], 'severe_percent': [100 * p1, 100 * p0],
    })

    result = {
        'hypothesis': name, 'group1': label1, 'group0': label0,
        'value1': 100 * p1, 'value0': 100 * p0,
        'difference': 100 * (p1 - p0), 'unit': 'percentage points',
        'statistic': float(z), 'test': 'one-sided two-proportion Z',
        'p_value': float(norm.sf(z)), 'log_p_value': float(norm.logsf(z)),
    }
    return table, result

speed_table, h1 = compare_shares(
    df, df['speed_limit'].isin([50, 60, 70]), df['speed_limit'].isin([20, 30]),
    'H1', '50–70 mph', '20–30 mph',
)
light_table, h2 = compare_shares(
    df, df['light_conditions'].eq(6), df['light_conditions'].eq(1),
    'H2', 'Без освещения', 'Дневной свет',
)
speed_table.to_csv(RESULTS / 'speed.csv', index=False)
light_table.to_csv(RESULTS / 'light.csv', index=False)
display(speed_table)
display(light_table)


In [ ]:
timed = df.dropna(subset=['date', 'hour']).copy()
timed['hour'] = timed['hour'].astype(int)
timed['peak'] = timed['hour'].isin([7, 8, 9, 16, 17, 18])

calendar = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31')
missing_days = calendar.difference(pd.DatetimeIndex(timed['date'].unique()))
if len(missing_days):
    raise ValueError(
        f'За {len(missing_days)} дней нет корректных наблюдений. '
        'Проверьте полноту периода. Отсутствующие данные нельзя автоматически считать нулём.'
    )
daily = timed.groupby(['date', 'peak']).size().unstack(fill_value=0)
daily = daily.reindex(index=calendar, columns=[True, False], fill_value=0)
daily.columns = ['peak_count', 'other_count']
daily.index.name = 'date'

daily['peak_per_hour'] = daily['peak_count'] / 6
daily['other_per_hour'] = daily['other_count'] / 18
daily['difference'] = daily['peak_per_hour'] - daily['other_per_hour']
difference = daily['difference']
if len(difference) < 30 or difference.std(ddof=1) == 0:
    raise ValueError('Недостаточно дней или вариации для проверки H3.')

test3 = ttest_1samp(difference, popmean=0, alternative='greater')
h3 = {
    'hypothesis': 'H3', 'group1': 'Часы пик', 'group0': 'Другие часы',
    'value1': float(daily['peak_per_hour'].mean()),
    'value0': float(daily['other_per_hour'].mean()),
    'difference': float(difference.mean()), 'unit': 'collisions/hour',
    'statistic': float(test3.statistic), 'test': 'one-sided paired daily t-test',
    'p_value': float(test3.pvalue),
}
daily.to_csv(RESULTS / 'daily.csv')
hourly = timed.groupby('hour').size().reindex(range(24), fill_value=0).rename('collision_count')
hourly.to_csv(RESULTS / 'hourly.csv')
heatmap = timed.groupby([timed['date'].dt.dayofweek.rename('weekday'), 'hour']).size().unstack(fill_value=0)
heatmap = heatmap.reindex(index=range(7), columns=range(24), fill_value=0)
heatmap.to_csv(RESULTS / 'heatmap.csv')
display(daily[['peak_per_hour', 'other_per_hour']].mean().to_frame('ДТП за час'))


In [ ]:
hypotheses = pd.DataFrame([h1, h2, h3])

hypotheses['p_adjusted'] = np.minimum(hypotheses['p_value'] * 3, 1)
hypotheses['decision'] = np.where(
    hypotheses['p_adjusted'] < ALPHA,
    'Отклоняем H0 в пользу направленной гипотезы',
    'Недостаточно оснований отклонить H0',
)
hypotheses.to_csv(RESULTS / 'hypotheses.csv', index=False)
display(hypotheses[['hypothesis', 'value1', 'value0', 'difference', 'unit', 'p_adjusted', 'decision']])
print('p-value, записанное как 0, означает численное округление очень малого значения.')


## 4. Корреляции и описательная регрессия

Пирсон для двух бинарных признаков совпадает с phi; сравниваются те же группы скорости и освещения. Числовые коды категорий не трактуются как непрерывные величины. Регуляризованная логистическая регрессия учитывает группу скорости, освещение, год и подразделение полиции. `exp(coefficient)` — отношение шансов при прочих включённых признаках; p-value коэффициентов не рассчитываются. Это ассоциации, а не причинные эффекты или процентное изменение вероятности.

In [ ]:
def calculate_phi(subset, column, positive_values):
    subset = subset.dropna(subset=['severe'])
    binary = subset[column].isin(positive_values).astype(int)
    if binary.nunique() > 1 and subset['severe'].nunique() > 1:
        coefficient = binary.corr(subset['severe'])
    else:
        coefficient = np.nan
    return len(subset), coefficient

speed_subset = df[df['speed_limit'].isin([20, 30, 50, 60, 70])]
light_subset = df[df['light_conditions'].isin([1, 6])]
speed_n, speed_phi = calculate_phi(speed_subset, 'speed_limit', [50, 60, 70])
light_n, light_phi = calculate_phi(light_subset, 'light_conditions', [6])
correlations = [
    {'feature': 'high_speed', 'n': speed_n, 'phi': speed_phi},
    {'feature': 'no_lighting', 'n': light_n, 'phi': light_phi},
]
correlation_table = pd.DataFrame(correlations)
correlation_table.to_csv(RESULTS / 'correlations.csv', index=False)
display(correlation_table)

model_data = df[
    df['speed_limit'].isin([20, 30, 50, 60, 70])
    & df['light_conditions'].isin([1, 6])
    & df['severe'].notna() & df['police_force'].gt(0)
].copy()
model_data['high_speed'] = model_data['speed_limit'].isin([50, 60, 70]).astype(int)
model_data['no_lighting'] = model_data['light_conditions'].eq(6).astype(int)

X_description = pd.get_dummies(
    model_data[['high_speed', 'no_lighting', 'collision_year', 'police_force']],
    columns=['collision_year', 'police_force'], drop_first=True, dtype=int,
)
y_description = model_data['severe'].astype(int)
if y_description.nunique() != 2:
    raise ValueError('Для регрессии нужны оба класса тяжести.')
descriptive_model = LogisticRegression(max_iter=2000)
descriptive_model.fit(X_description, y_description)

regression = pd.DataFrame({
    'feature': X_description.columns,
    'coefficient': descriptive_model.coef_[0],
    'odds_ratio': np.exp(descriptive_model.coef_[0]),
})
regression.to_csv(RESULTS / 'regression.csv', index=False)
print('Наблюдений в описательной регрессии:', len(model_data))
display(regression.head(2))


## 5. Классификация тяжести ДТП

Обучение: 2022–2023; тест: 2024. Признаки — скорость, освещение, подразделение полиции, день недели и час. Категории кодируются One-Hot, неизвестные значения допускаются, пропуски становятся категорией `unknown`. Преобразования обучаются только на train. Логистическая регрессия балансирует классы, порог — 0,5; настройки по тесту не подбираются. Baseline всегда выбирает наиболее частый класс train.

Оцениваются accuracy, precision, recall, F1, ROC-AUC и AP, матрица ошибок и ROC/PR-кривые. Ориентир AP — доля тяжёлых ДТП в тесте. Балансировка требует отдельной калибровки вероятностей; модель оценивает тяжесть уже зарегистрированного ДТП, а не вероятность возникновения ДТП.

In [ ]:
FEATURES = ['speed_limit', 'light_conditions', 'police_force', 'weekday', 'hour']
predictive_data = df[df['severe'].notna()].copy()
X_predictive = predictive_data[FEATURES].copy()
for column in FEATURES:
    values = pd.to_numeric(X_predictive[column], errors='coerce')
    values = values.where(values.ge(0) & values.mod(1).eq(0))
    X_predictive[column] = values.astype('Int64').astype('string').fillna('unknown').astype(str)
y_predictive = predictive_data['severe'].astype(int)

train_mask = predictive_data['collision_year'].between(2022, 2023)
test_mask = predictive_data['collision_year'].eq(2024)
X_train, X_test = X_predictive.loc[train_mask], X_predictive.loc[test_mask]
y_train, y_test = y_predictive.loc[train_mask], y_predictive.loc[test_mask]
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Обучающий и тестовый периоды должны содержать оба класса.')

encoder = OneHotEncoder(handle_unknown='ignore')

classifier = Pipeline([
    ('features', encoder),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
])
classifier.fit(X_train, y_train)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

probabilities = classifier.predict_proba(X_test)[:, 1]

predictions = (probabilities >= 0.5).astype(int)
baseline_probabilities = baseline.predict_proba(X_test)[:, 1]
baseline_predictions = baseline.predict(X_test)

def score_model(name, actual, predicted, scores):
    return {
        'model': name, 'test_rows': len(actual),
        'accuracy': accuracy_score(actual, predicted),
        'precision': precision_score(actual, predicted, zero_division=0),
        'recall': recall_score(actual, predicted, zero_division=0),
        'f1': f1_score(actual, predicted, zero_division=0),
        'roc_auc': roc_auc_score(actual, scores),
        'average_precision': average_precision_score(actual, scores),
    }
ml_metrics = pd.DataFrame([
    score_model('Baseline: most frequent', y_test, baseline_predictions, baseline_probabilities),
    score_model('Logistic regression', y_test, predictions, probabilities),
])
ml_metrics.to_csv(RESULTS / 'ml_metrics.csv', index=False)
print(f'Обучение: {len(y_train):,}. Тест: {len(y_test):,}. Доля тяжёлых в тесте: {y_test.mean():.2%}')
display(ml_metrics)
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
pd.DataFrame(cm, index=['actual_light', 'actual_severe'],
             columns=['predicted_light', 'predicted_severe']).to_csv(RESULTS / 'confusion_matrix.csv')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(cm, display_labels=['Лёгкое', 'Тяжёлое']).plot(ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, probabilities, ax=axes[1], name='Logistic regression')
PrecisionRecallDisplay.from_predictions(y_test, probabilities, ax=axes[2], name='Logistic regression')
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Доля положительного класса')
axes[2].legend()
finish_figure(fig, 'ml_evaluation.png', show=True)


## 6. Графики для исследования и отчёта

HTML-файлы включают Plotly и открываются отдельно без Python. Есть подсказки, масштабирование и переключатель года у сравнений скорости/освещения; для вывода в ноутбуке могут потребоваться доверие к файлу и доступ к CDN. Доли рассчитываются среди ДТП с известной тяжестью каждой группы. Переключатель пересчитывает числитель и знаменатель. Число ДТП без данных о транспортном потоке не является риском на поездку. PNG сохраняются отдельно для отчёта.

In [ ]:
def export_interactive(fig, filename):
    fig.write_html(RESULTS / filename, include_plotlyjs=True, full_html=True)
    display(fig)

fig_annual = px.line(annual, x='year', y='collision_count', markers=True,
                    title='Количество зарегистрированных ДТП по годам',
                    labels={'year': 'Год', 'collision_count': 'Количество ДТП'})
export_interactive(fig_annual, 'annual_interactive.html')

fig_hourly = px.bar(hourly.reset_index(), x='hour', y='collision_count',
                   title=f'Распределение ДТП по часам, {START_YEAR}–{END_YEAR}',
                   labels={'hour': 'Час', 'collision_count': 'Количество ДТП'})
export_interactive(fig_hourly, 'hourly_interactive.html')

fig_heatmap = go.Figure(go.Heatmap(
    z=heatmap.values, x=list(range(24)),
    y=['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'], colorscale='Blues',
    colorbar={'title': 'ДТП'},
))
fig_heatmap.update_layout(title='ДТП по дням недели и часам', xaxis_title='Час')
export_interactive(fig_heatmap, 'heatmap_interactive.html')

def yearly_share_chart(kind):

    if kind == 'speed':
        column = 'speed_limit'
        groups = [[50, 60, 70], [20, 30]]
        labels = ['50–70 mph', '20–30 mph']
        comparison_title = 'по ограничению скорости'
    else:
        column = 'light_conditions'
        groups = [[6], [1]]
        labels = ['Без освещения', 'Дневной свет']
        comparison_title = 'по освещению'
    fig = go.Figure()
    years = ['Весь период', *sorted(df['collision_year'].dropna().astype(int).unique())]
    exports = []

    for i, year in enumerate(years):
        subset = df if year == 'Весь период' else df[df['collision_year'].eq(year)]
        masks = [subset[column].isin(groups[0]), subset[column].isin(groups[1])]
        values, details = [], []
        for label, mask in zip(labels, masks):
            known = subset.loc[mask, 'severe'].dropna()
            share = float(known.mean() * 100) if len(known) else None
            values.append(share)
            details.append([len(known), int(known.sum())])
            exports.append({'period': str(year), 'group': label, 'total': len(known),
                            'severe_count': int(known.sum()), 'severe_percent': share})
        fig.add_trace(go.Bar(
            x=labels, y=values, customdata=details, visible=(i == 0), name=str(year),
            hovertemplate='%{x}<br>Доля: %{y:.2f}%<br>Всего: %{customdata[0]}<br>Тяжёлых: %{customdata[1]}<extra></extra>',
        ))
    buttons = [dict(label=str(year), method='update',
                    args=[{'visible': [j == i for j in range(len(years))]}])
               for i, year in enumerate(years)]
    fig.update_layout(
        title='Доля тяжёлых и смертельных ДТП ' + comparison_title,
        yaxis_title='Доля, %', updatemenus=[{'buttons': buttons, 'x': 1.0, 'y': 1.2}],
        yaxis_range=[0, 100], showlegend=False,
    )
    pd.DataFrame(exports).to_csv(RESULTS / f'{kind}_by_year.csv', index=False)
    return fig

export_interactive(yearly_share_chart('speed'), 'speed_interactive.html')
export_interactive(yearly_share_chart('light'), 'light_interactive.html')
print('HTML-графики сохранены в', RESULTS)


In [ ]:
for table, name, title in [
    (speed_table, 'speed.png', 'Доля тяжёлых ДТП по ограничению скорости'),
    (light_table, 'light.png', 'Доля тяжёлых ДТП по освещению'),
]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(table['group'], table['severe_percent'])
    ax.set(ylabel='Доля, %', title=title)
    finish_figure(fig, name)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(annual['year'], annual['collision_count'])
ax.set(xlabel='Год', ylabel='Количество ДТП', title='Годовая динамика')
finish_figure(fig, 'annual.png')
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(hourly.index, hourly.values)
ax.set(xlabel='Час', ylabel='Количество ДТП', title='ДТП по часам', xticks=range(24))
finish_figure(fig, 'hourly.png')
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap.values, aspect='auto', cmap='Blues')
ax.set(xticks=range(24), yticks=range(7), yticklabels=['Пн','Вт','Ср','Чт','Пт','Сб','Вс'], xlabel='Час')
fig.colorbar(im, ax=ax, label='Количество ДТП')
finish_figure(fig, 'heatmap.png')


## 7. Представление и дашборд Superset

Настройка и запуск Superset описаны в [README, разделы 5–7](README.md#5-запуск-superset-на-mac). Для отдельного Compose на Mac: `host.docker.internal`, порт из `.env`, база `road_safety`, пользователь `road_user`, фактический пароль. В одной сети Compose используйте `postgres:5432`. Ноутбук создаёт представление, а дашборд собирается отдельно:

1. Добавьте `public.collisions_dashboard` в **Data → Datasets**; `collision_date` — временное поле, остальные категории доступны для группировки/фильтров.
2. Количество: `COUNT(*)`; доля тяжёлых: `100.0 * SUM(severe) / NULLIF(COUNT(severe), 0)` — только известная тяжесть.
3. Создайте годовую динамику, доли по `speed_group` / `lighting_group`, распределение по `hour` и тепловую карту `weekday × hour`. Исключите пустые часы/дни; для долей выберите сравниваемые группы.
4. Добавьте фильтры года, скорости и освещения. Исследовательским графикам задайте 2022–2024; годовой динамике оставьте весь период. Проверьте изменение показателей, сохраните дашборд «Анализ ДТП», экспортируйте ZIP в `superset/dashboard_export.zip`.

Фильтры Superset не пересчитывают статистические тесты и ML из Python. Следующая ячейка выполняет `CREATE OR REPLACE VIEW`, не копируя строки таблицы.

In [ ]:
view_sql = """
CREATE OR REPLACE VIEW public.collisions_dashboard AS
SELECT
    collision_index, collision_year, date AS collision_date,
    EXTRACT(HOUR FROM time)::integer AS hour,
    EXTRACT(ISODOW FROM date)::integer AS weekday,
    speed_limit, light_conditions, police_force, collision_severity,
    CASE WHEN collision_severity IN (1, 2) THEN 1
         WHEN collision_severity = 3 THEN 0 END AS severe,
    CASE WHEN speed_limit IN (50, 60, 70) THEN '50–70 mph'
         WHEN speed_limit IN (20, 30) THEN '20–30 mph'
         WHEN speed_limit = 40 THEN '40 mph'
         ELSE 'Неизвестно/другое' END AS speed_group,
    CASE WHEN light_conditions = 1 THEN 'Дневной свет'
         WHEN light_conditions = 6 THEN 'Без освещения'
         ELSE 'Другие/неизвестные условия' END AS lighting_group
FROM public.collisions
"""
with engine.begin() as conn:
    conn.execute(text(view_sql))
print('Представление public.collisions_dashboard готово для Superset.')


## 8. Выводы

Выводы ниже вычисляются из текущих таблиц и записываются в `results/conclusions.md`. Ограничения сохраняются независимо от p-value и метрик. Полноту ETL подтверждают отчёт и БД, готовность Superset — сохранённый дашборд, фильтры и экспорт. Дополнительные внешние данные не используются. Исходный CSV и `.env` остаются локально; версии библиотек записываются в `analysis_environment.json`.

Источники: [STATS19 / DfT](https://www.gov.uk/government/statistical-data-sets/road-safety-open-data), [PostgreSQL Docker](https://hub.docker.com/_/postgres), [Docker Compose](https://docs.docker.com/compose/), [Psycopg COPY](https://www.psycopg.org/psycopg3/docs/basic/copy.html), [Superset](https://superset.apache.org/user-docs/6.0.0/quickstart/), [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). Коды данных сверяйте со словарём используемой версии.

In [ ]:
lines = [
    '# Результаты анализа ДТП', '',
    f'В PostgreSQL {total:,} записей. В периоде {START_YEAR}–{END_YEAR} — {len(df):,}.', '',
]
for row in hypotheses.itertuples():
    unit = 'процентных пункта' if row.unit == 'percentage points' else 'ДТП в час'
    p_text = '< 1e-300 (численное округление)' if row.p_adjusted == 0 else f'{row.p_adjusted:.3g}'
    lines.append(
        f'- {row.hypothesis}: {row.group1} — {row.value1:.2f}, '
        f'{row.group0} — {row.value0:.2f}. Разница {row.difference:.2f} {unit}. '
        f'Скорректированное p-value {p_text}. {row.decision}.'
    )
lines += ['', '## Классификация на 2024 год', '']
for row in ml_metrics.itertuples():
    lines.append(
        f'- {row.model}: accuracy {row.accuracy:.3f}, precision {row.precision:.3f}, '
        f'recall {row.recall:.3f}, F1 {row.f1:.3f}, '
        f'ROC-AUC {row.roc_auc:.3f}, AP {row.average_precision:.3f}.'
    )
lines += [
    '', '## Ограничения', '',
    '- STATS19 содержит зарегистрированные полицией ДТП с пострадавшими. Изменения регистрации и полноты данных ограничивают сравнение лет.',
    '- Ограничение скорости отличается от фактической скорости автомобиля.',
    '- Без данных о транспортном потоке число ДТП нельзя интерпретировать как риск на поездку.',
    '- Статистические тесты предполагают условия независимости. Временная и пространственная зависимость могут влиять на значимость.',
    '- Наблюдаемые связи не устанавливают причинность. В модели отсутствуют многие характеристики дороги, автомобиля и участника.',
    '- Классификация оценивает тяжесть среди уже зарегистрированных ДТП. Предсказание самого возникновения ДТП не исследуется.',
    '- Метрики теста характеризуют период 2024 года. Балансированная модель требует отдельной калибровки перед интерпретацией вероятностей как абсолютных оценок.',
]
conclusions = '\n'.join(lines)
(RESULTS / 'conclusions.md').write_text(conclusions, encoding='utf-8')
display(Markdown(conclusions))
print('Результаты сохранены в', RESULTS)
engine.dispose()
